# Prevendo a Copa do Mundo 2026 com Machine Learning

## O que este projeto faz?

Este notebook usa **dados históricos de todos os jogos da Copa do Mundo desde 1930** para construir um modelo que prevê o resultado de cada partida.

Com esse modelo, simulamos o torneio completo **10.000 vezes** e respondemos à pergunta:

> **Qual seleção tem mais chance de ser campeã da Copa 2026?**

---

## Fluxo do projeto

```
Dados históricos (1930–2022)
        ↓
  Calcular ELO de cada seleção
        ↓
  Treinar modelo: ELO do time A vs ELO do time B → quem ganha?
        ↓
  Simular Copa 2026 dez mil vezes (Monte Carlo)
        ↓
  Resultado: probabilidade de cada seleção ser campeã
```

## 1. Importações

Bibliotecas que vamos usar:
- **pandas / numpy**: manipulação de dados
- **matplotlib / seaborn**: gráficos
- **xgboost**: algoritmo de Machine Learning (vamos ver o que é mais adiante)
- **sklearn**: utilitários de avaliação do modelo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

DATA = Path('data/fjelstul')

print('Tudo importado com sucesso!')

---

## 2. Carregando os dados históricos

Usamos o banco de dados do **Fjelstul World Cup Database** (GitHub), que contém todos os jogos da Copa do Mundo de 1930 a 2022 com informações detalhadas: placar, fase, estádio, etc.

Cada linha é um jogo. As colunas que nos interessam são:
- `home_team_name` / `away_team_name`: as duas seleções
- `home_team_score` / `away_team_score`: o placar
- `result`: quem ganhou (`home team win`, `away team win`, `draw`)

In [ ]:
# Carrega todos os jogos
matches = pd.read_csv(DATA / 'matches.csv', low_memory=False)

# Converte data para tipo datetime (facilita ordenação)
matches['match_date'] = pd.to_datetime(matches['match_date'])

# Mantém só as colunas que precisamos
cols = [
    'match_id', 'match_date', 'tournament_name',
    'stage_name', 'group_stage', 'knockout_stage',
    'home_team_name', 'away_team_name',
    'home_team_score', 'away_team_score',
    'home_team_win', 'away_team_win', 'draw', 'result'
]
matches = matches[cols].sort_values('match_date').reset_index(drop=True)

print(f'Total de jogos: {len(matches)}')
print(f'Período: {matches.match_date.min().year} a {matches.match_date.max().year}')
print(f'Seleções únicas: {pd.concat([matches.home_team_name, matches.away_team_name]).nunique()}')
matches.tail(5)

In [ ]:
# Distribuição dos resultados
result_counts = matches['result'].value_counts()

fig, ax = plt.subplots()
result_counts.plot(kind='bar', ax=ax, color=['#1a9641', '#d9d9d9', '#d7191c'])
ax.set_title('Distribuição de resultados na Copa do Mundo (1930–2022)')
ax.set_xlabel('')
ax.set_ylabel('Número de jogos')
ax.set_xticklabels(['Vitória do mandante', 'Vitória do visitante', 'Empate'], rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center')
plt.tight_layout()
plt.show()

print('\nObs: vitórias do mandante são mais comuns ("vantagem de jogar em casa")')
print('Este desequilíbrio vai afetar o modelo — voltaremos a isso.')

---

## 3. O sistema ELO — medindo a força de cada seleção

### Por que não usar simplesmente a taxa de vitórias?

Se o Brasil vence 8 de 10 jogos e a Bolívia também vence 8 de 10, eles teriam o mesmo "win rate". Mas os adversários do Brasil são muito mais fortes. **A taxa de vitórias ignora a qualidade do adversário.**

### Como o ELO funciona?

Cada seleção começa com **1500 pontos**. Após cada jogo, os pontos são atualizados com base em duas coisas:
1. **O resultado real** (ganhou, empatou ou perdeu)
2. **O resultado esperado** (calculado pela diferença de ELO entre os times)

**Fórmula:**

```
Resultado esperado do Time A:
  E_A = 1 / (1 + 10^((ELO_B - ELO_A) / 400))

Novo ELO do Time A:
  ELO_A_novo = ELO_A + K × (resultado_real - E_A)

  resultado_real: 1 = vitória, 0.5 = empate, 0 = derrota
  K = 30 (o quanto cada jogo movimenta o rating)
```

**Exemplo prático:**
- Brasil (ELO 2000) joga contra Japão (ELO 1500)
- O modelo espera que o Brasil vença com ~97% de probabilidade
- Se o Brasil ganha: ganha apenas ~1 ponto (era esperado)
- Se o Japão ganha: Japão ganha ~29 pontos (grande surpresa!)

In [ ]:
def expected_score(elo_a, elo_b):
    """Probabilidade esperada de vitória do time A dado os ELOs dos dois times."""
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, score_a, k=30):
    """
    Atualiza o ELO do time A após um jogo.
    score_a: 1 = vitória, 0.5 = empate, 0 = derrota
    Retorna o novo ELO do time A.
    """
    e_a = expected_score(elo_a, elo_b)
    return elo_a + k * (score_a - e_a)

# Demonstração: o que acontece com o ELO em diferentes cenários?
cenarios = [
    ('Brasil (forte)',  2000, 'Japão (médio)',    1500, 1.0,  'Brasil vence'),
    ('Brasil (forte)',  2000, 'Japão (médio)',    1500, 0.5,  'Empate'),
    ('Brasil (forte)',  2000, 'Japão (médio)',    1500, 0.0,  'Japão vence'),
    ('França (forte)', 2000, 'Alemanha (forte)', 1950, 1.0,  'França vence'),
]

# Usa lista para evitar f-strings aninhadas (compatível com Python 3.10)
colunas = ['Cenário', 'ELO A antes', 'ELO B antes', 'ELO A depois', 'Variação']
print(f'{colunas[0]:<35} {colunas[1]:>11} {colunas[2]:>11} {colunas[3]:>12} {colunas[4]:>9}')
print('-' * 80)
for nome_a, elo_a, nome_b, elo_b, score, desc in cenarios:
    novo = update_elo(elo_a, elo_b, score)
    print(f'{desc:<35} {elo_a:>11} {elo_b:>11} {novo:>12.1f} {novo-elo_a:>+9.1f}')

In [ ]:
# Calculamos o ELO de todas as seleções passando por todos os jogos em ordem cronológica

elo_ratings = {}  # dicionário: nome da seleção → ELO atual
ELO_INICIAL = 1500
K = 30

# Vamos guardar o ELO de cada time ANTES de cada jogo (para usar como feature do modelo)
elo_antes_home = []
elo_antes_away = []

for _, jogo in matches.iterrows():
    home = jogo['home_team_name']
    away = jogo['away_team_name']

    # Se a seleção é nova (primeira Copa), começa em 1500
    elo_h = elo_ratings.get(home, ELO_INICIAL)
    elo_a = elo_ratings.get(away, ELO_INICIAL)

    # Guarda o ELO ANTES do jogo (sem "ver o futuro")
    elo_antes_home.append(elo_h)
    elo_antes_away.append(elo_a)

    # Define o resultado do ponto de vista do time da casa
    if jogo['home_team_win']:
        score_home = 1.0  # vitória
    elif jogo['draw']:
        score_home = 0.5  # empate
    else:
        score_home = 0.0  # derrota

    # Atualiza o ELO dos dois times
    elo_ratings[home] = update_elo(elo_h, elo_a, score_home, K)
    elo_ratings[away] = update_elo(elo_a, elo_h, 1 - score_home, K)  # resultado inverso

# Adiciona os ELOs como colunas do dataframe
matches['elo_home'] = elo_antes_home
matches['elo_away'] = elo_antes_away
matches['elo_diff'] = matches['elo_home'] - matches['elo_away']  # diferença de força

print(f'ELO calculado para {len(elo_ratings)} seleções')
print(f'\nTop 10 seleções por ELO ao final de 2022:')
ranking = pd.Series(elo_ratings).sort_values(ascending=False)
print(ranking.head(10).round(1).to_string())

In [ ]:
# Visualização: top 20 seleções por ELO
top20 = ranking.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['gold' if i == 0 else 'silver' if i == 1 else '#cd7f32' if i == 2 else '#4472C4'
          for i in range(len(top20))]
bars = ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
ax.set_xlabel('ELO Rating')
ax.set_title('Ranking ELO das Seleções — calculado com dados da Copa (1930–2022)')
ax.axvline(1500, color='red', linestyle='--', alpha=0.5, label='ELO médio (1500)')
ax.legend()
for bar, val in zip(bars[::-1], top20.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Evolução histórica do ELO do Brasil ao longo das Copas
brasil_jogos = matches[
    (matches['home_team_name'] == 'Brazil') | (matches['away_team_name'] == 'Brazil')
].copy()

# Pega o ELO do Brasil em cada jogo (seja como mandante ou visitante)
brasil_jogos['elo_brasil'] = brasil_jogos.apply(
    lambda r: r['elo_home'] if r['home_team_name'] == 'Brazil' else r['elo_away'], axis=1
)

fig, ax = plt.subplots()
ax.plot(brasil_jogos['match_date'], brasil_jogos['elo_brasil'], color='#009c3b', linewidth=2)
ax.fill_between(brasil_jogos['match_date'], 1500, brasil_jogos['elo_brasil'],
                alpha=0.15, color='#009c3b')
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Evolução do ELO do Brasil nas Copas do Mundo (1930–2022)')
ax.set_xlabel('Ano')
ax.set_ylabel('ELO Rating')

# Marca os anos de título
for ano, titulo in [(1958, '1958'), (1962, '1962'), (1970, '1970'), (1994, '1994'), (2002, '2002')]:
    idx = brasil_jogos[brasil_jogos['match_date'].dt.year == ano].index
    if len(idx):
        ax.axvline(pd.Timestamp(f'{ano}-01-01'), color='gold', alpha=0.6, linewidth=1.5)
        ax.text(pd.Timestamp(f'{ano}-01-01'), ax.get_ylim()[0] + 20, f'🏆{titulo}',
                fontsize=7, color='goldenrod')

plt.tight_layout()
plt.show()

---

## 4. Construindo o modelo de previsão de resultado

### O que o modelo vai aprender?

Para cada jogo, o modelo recebe:
- ELO do time da casa antes do jogo
- ELO do time visitante antes do jogo
- A diferença entre eles (feature derivada)

E precisa prever:
- **0** = vitória do time da casa
- **1** = empate
- **2** = vitória do visitante

### Que algoritmo usamos? XGBoost

XGBoost (eXtreme Gradient Boosting) é uma família de algoritmos baseada em **árvores de decisão**. Imagine uma série de perguntas:

```
"A diferença de ELO é maior que 200?"
  → Sim: "O time da casa é favorito?"
    → Sim: provável vitória da casa
    → Não: ...
  → Não: ...
```

O XGBoost treina centenas dessas árvores em sequência, onde cada nova árvore tenta corrigir os erros da anterior. É um dos algoritmos mais usados em competições de Machine Learning.

In [ ]:
# Prepara o dataset de treino

# Converte resultado textual em número
result_map = {'home team win': 0, 'draw': 1, 'away team win': 2}
matches['result_label'] = matches['result'].map(result_map)

# Features que o modelo vai usar
FEATURES = ['elo_home', 'elo_away', 'elo_diff']

# Remove linhas com dado faltante
df_model = matches.dropna(subset=FEATURES + ['result_label']).copy()

X = df_model[FEATURES]
y = df_model['result_label'].astype(int)

# Separa 80% para treino e 20% para teste
# stratify=y garante que a proporção de cada resultado seja igual nos dois conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Jogos para treino: {len(X_train)}')
print(f'Jogos para teste:  {len(X_test)}')
print(f'\nDistribuição no treino:')
print(y_train.value_counts().rename({0: 'Vitória casa', 1: 'Empate', 2: 'Vitória visitante'}))

In [ ]:
# Treina o modelo XGBoost
#
# Parâmetros importantes:
#   n_estimators: quantas árvores o modelo vai construir
#   max_depth: quão profunda pode ser cada árvore (complexidade)
#   learning_rate: o quanto cada árvore contribui (menor = mais conservador)
#   objective='multi:softprob': classifica em múltiplas classes e retorna probabilidades

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    early_stopping_rounds=20,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False  # muda para True se quiser ver o progresso do treino
)

print('Modelo treinado!')
print(f'Melhor número de árvores: {model.best_iteration}')

### Avaliando o modelo

Usamos o conjunto de teste (jogos que o modelo **nunca viu**) para avaliar a qualidade das previsões.

**Métricas importantes:**
- **Precision**: dos jogos que o modelo previu como vitória, quantos % realmente foram vitória?
- **Recall**: dos jogos que realmente foram vitória, quantos % o modelo acertou?
- **F1-score**: média harmônica de precision e recall (equilíbrio entre os dois)

In [ ]:
preds = model.predict(X_test)
nomes_classes = ['Vitória casa (0)', 'Empate (1)', 'Vitória visitante (2)']

print('=== Relatório de avaliação ===')
print(classification_report(y_test, preds, target_names=nomes_classes, zero_division=0))

# Matriz de confusão: mostra onde o modelo erra
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, preds,
    display_labels=['Casa', 'Empate', 'Visitante'],
    cmap='Blues', ax=ax
)
ax.set_title('Matriz de Confusão\n(linhas = real, colunas = previsto)')
plt.tight_layout()
plt.show()

print('\nInterpretação da matriz:')
print('  Diagonal principal = acertos')
print('  Fora da diagonal   = erros')

In [ ]:
# Visualiza como a diferença de ELO afeta a probabilidade de vitória
# Isso é a "inteligência" do modelo explicada visualmente

diffs = np.linspace(-500, 500, 200)
elo_base = 1600  # ELO base do time da casa

linhas = pd.DataFrame({
    'elo_home': elo_base,
    'elo_away': elo_base - diffs,
    'elo_diff': diffs
})

probs = model.predict_proba(linhas)

fig, ax = plt.subplots()
ax.plot(diffs, probs[:, 0], label='Vitória do mandante', color='#1a9641', linewidth=2)
ax.plot(diffs, probs[:, 1], label='Empate',              color='#d9d9d9', linewidth=2)
ax.plot(diffs, probs[:, 2], label='Vitória do visitante', color='#d7191c', linewidth=2)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, label='Times com mesmo ELO')
ax.set_xlabel('Diferença de ELO (mandante − visitante)')
ax.set_ylabel('Probabilidade')
ax.set_title('O que o modelo aprende: ELO → probabilidade de resultado')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

print('Quando elo_diff = 0 (times iguais), o resultado é incerto.')
print('Quando elo_diff cresce, o modelo favorece mais o mandante.')

---

## 5. Simulação Monte Carlo do torneio

### Por que Monte Carlo?

A Copa do Mundo é um torneio em cascata: quem você enfrenta nas quartas de final depende de quem venceu nos grupos e nas oitavas. Não existe uma fórmula matemática simples para calcular "a probabilidade do Brasil ser campeão" considerando todos os caminhos possíveis.

A solução é simular o torneio inteiro do começo ao fim, muitas vezes:

```
Simulação 1:  Brasil bate a França → França elimina → Brasil campeão
Simulação 2:  Argentina elimina o Brasil nas quartas
Simulação 3:  Brasil campeão
...
Simulação 10.000: Brasil campeão

Resultado: Brasil foi campeão em 2.800 das 10.000 simulações → 28% de chance
```

Isso se chama **simulação de Monte Carlo** — o nome vem do famoso cassino de Mônaco, porque usa aleatoriedade controlada para estimar probabilidades.

### Como simulamos um jogo?

O modelo nos dá probabilidades: ex. `{casa: 55%, empate: 25%, visitante: 20%}`

Para simular, geramos um número aleatório entre 0 e 1:
- Se cair entre 0.00 e 0.55 → vitória do mandante
- Se cair entre 0.55 e 0.80 → empate
- Se cair entre 0.80 e 1.00 → vitória do visitante

### Empate no mata-mata

A partir das oitavas de final, **empate não existe como resultado final**. Se o modelo prevê empate, simulamos pênaltis — que são essencialmente um **sorteio 50/50** (na prática, estatísticas mostram que o desempenho em pênaltis é muito imprevisível).

In [ ]:
def prever_jogo(model, elo_home, elo_away):
    """
    Retorna as probabilidades de resultado de um jogo.
    Usa numpy array em vez de DataFrame — muito mais rápido dentro de loops.
    Saída: {'home': prob_vitória_casa, 'draw': prob_empate, 'away': prob_vitória_visitante}
    """
    X = np.array([[elo_home, elo_away, elo_home - elo_away]])
    probs = model.predict_proba(X)[0]
    return {'home': probs[0], 'draw': probs[1], 'away': probs[2]}


def simular_jogo(probs, mata_mata=False):
    """
    Sorteia o vencedor de um jogo com base nas probabilidades.
    Se mata_mata=True e der empate, decide nos pênaltis (50/50).
    Retorna 'home' ou 'away'.
    """
    sorteio = np.random.random()

    if sorteio < probs['home']:
        return 'home'
    elif sorteio < probs['home'] + probs['draw']:
        if mata_mata:
            # Empate no mata-mata → pênaltis = 50/50
            return 'home' if np.random.random() < 0.5 else 'away'
        return 'draw'
    else:
        return 'away'


print('Funções de simulação definidas!')

# Exemplo: simulando Brasil x Argentina
elo_brasil    = elo_ratings.get('Brazil',    1600)
elo_argentina = elo_ratings.get('Argentina', 1600)

probs = prever_jogo(model, elo_brasil, elo_argentina)
print(f'\nBrasil (ELO {elo_brasil:.0f}) x Argentina (ELO {elo_argentina:.0f})')
print(f'  Probabilidade de vitória do Brasil:    {probs["home"]:5.1%}')
print(f'  Probabilidade de empate:               {probs["draw"]:5.1%}')
print(f'  Probabilidade de vitória da Argentina: {probs["away"]:5.1%}')

print('\n5 simulações deste jogo (mata-mata):')
for i in range(5):
    resultado = simular_jogo(probs, mata_mata=True)
    vencedor = 'Brasil' if resultado == 'home' else 'Argentina'
    print(f'  Simulação {i+1}: {vencedor}')

### Montando o chaveamento da Copa 2026

A Copa 2026 tem 48 seleções, divididas em 12 grupos de 4. Avançam:
- Os 2 primeiros de cada grupo: **24 times**
- Os 8 melhores terceiros colocados: **8 times**
- Total na fase eliminatória: **32 times**

Como ainda não temos os dados em tempo real do Kaggle, vamos simular usando as **32 seleções com maior ELO** e um chaveamento hipotético. Quando os dados do Kaggle estiverem integrados, basta substituir este bloco.

In [ ]:
# As 32 seleções com maior ELO no final de 2022 (proxy para a Copa 2026)
# Quando tivermos dados reais do Kaggle, substituímos este bloco

top32 = ranking.head(32)
seleções_2026 = list(top32.index)

print('32 seleções na simulação:')
for i, (seleção, elo) in enumerate(top32.items(), 1):
    print(f'  {i:2d}. {seleção:<20} ELO: {elo:.0f}')

In [ ]:
# Pré-calcula as probabilidades para todos os pares possíveis de seleções.
# Como os ELOs são fixos (não mudam entre simulações), podemos calcular UMA VEZ
# e reutilizar nas 10.000 simulações — muito mais rápido.

from itertools import combinations

prob_cache = {}
for t1, t2 in combinations(seleções_2026, 2):
    e1 = elo_ratings.get(t1, 1500)
    e2 = elo_ratings.get(t2, 1500)
    prob_cache[(t1, t2)] = prever_jogo(model, e1, e2)  # t1 joga em casa
    prob_cache[(t2, t1)] = prever_jogo(model, e2, e1)  # t2 joga em casa

print(f'Probabilidades pré-calculadas para {len(prob_cache)} pares de times.')


def simular_torneio(seleções, prob_cache):
    """
    Simula o mata-mata da Copa do Mundo.
    Usa o cache de probabilidades em vez de chamar o modelo a cada jogo —
    isso torna as 10.000 simulações muito mais rápidas.

    A lógica:
    - Começa com N times
    - A cada rodada, times adjacentes se enfrentam: [0]x[1], [2]x[3], ...
    - Os vencedores avançam para a próxima fase
    - Repete até sobrar 1 time (o campeão)
    """
    times = list(seleções)
    np.random.shuffle(times)  # chaveamento aleatório a cada simulação

    while len(times) > 1:
        proxima_fase = []
        for i in range(0, len(times), 2):
            home = times[i]
            away = times[i + 1]
            probs = prob_cache[(home, away)]
            resultado = simular_jogo(probs, mata_mata=True)
            vencedor = home if resultado == 'home' else away
            proxima_fase.append(vencedor)
        times = proxima_fase

    return times[0]  # o campeão desta simulação


# Teste rápido com 5 simulações
print('\nTeste de 5 simulações:')
for i in range(5):
    campeão = simular_torneio(seleções_2026, prob_cache)
    print(f'  Simulação {i+1}: {campeão}')

In [ ]:
import time

N_SIMULACOES = 10_000

print(f'Rodando {N_SIMULACOES:,} simulações do torneio...')
np.random.seed(42)  # garante que os resultados sejam reproduzíveis

titulos = {}  # conta quantas vezes cada seleção foi campeã

inicio = time.time()
for _ in range(N_SIMULACOES):
    campeão = simular_torneio(seleções_2026, prob_cache)
    titulos[campeão] = titulos.get(campeão, 0) + 1
elapsed = time.time() - inicio

# Converte contagem em probabilidades
prob_campea = (
    pd.Series(titulos)
    .sort_values(ascending=False)
    .div(N_SIMULACOES)
    .rename('prob_campeã')
)

print(f'Concluído em {elapsed:.1f}s ({N_SIMULACOES/elapsed:.0f} simulações/segundo)')
print(f'Seleções que venceram ao menos uma simulação: {len(prob_campea)} de {len(seleções_2026)}')

In [ ]:
# Resultado final: probabilidade de ser campeão
print('=' * 45)
print('    PROBABILIDADE DE SER CAMPEÃO — Copa 2026')
print('=' * 45)
for i, (seleção, prob) in enumerate(prob_campea.head(15).items(), 1):
    barra = '█' * int(prob * 100)
    print(f'  {i:2d}. {seleção:<20} {prob:5.1%}  {barra}')
print('=' * 45)
print(f'\nTotal simulado: {N_SIMULACOES:,} torneios')

In [ ]:
# Gráfico final
top15 = prob_campea.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
cores = ['gold' if i == 0 else 'silver' if i == 1 else '#cd7f32' if i == 2 else '#4472C4'
         for i in range(len(top15))]

bars = ax.barh(top15.index[::-1], top15.values[::-1], color=cores[::-1])
ax.set_xlabel('Probabilidade de ser campeã')
ax.set_title(
    f'Copa do Mundo 2026 — Probabilidade de cada seleção ser campeã\n'
    f'(baseado em {N_SIMULACOES:,} simulações Monte Carlo + ELO histórico)'
)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

for bar, val in zip(bars[::-1], top15.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Resumo do projeto

| Etapa | O que fizemos |
|-------|---------------|
| **Dados** | 1.248 jogos da Copa (1930–2022) via Fjelstul World Cup Database |
| **ELO** | Sistema de rating que mede a força de cada seleção considerando a qualidade dos adversários |
| **Modelo** | XGBoost treinado para prever o resultado de um jogo a partir do ELO dos dois times |
| **Monte Carlo** | 10.000 simulações completas do torneio para estimar probabilidades |
| **Empates** | Tratados como resultado válido na fase de grupos; no mata-mata viram pênaltis (50/50) |

## Próximos passos

- **Integrar dados da Copa 2026** (Kaggle): atualizar o ELO com os resultados reais dos grupos
- **Usar o chaveamento real**: substituir o chaveamento aleatório pelo bracket oficial
- **Adicionar mais features**: posse de bola, chutes a gol, dados de elenco (idade média, valor de mercado)
- **Deploy no Streamlit**: interface interativa para qualquer pessoa simular cenários